# 1 — Fetch All BankNifty & Nifty Instruments

**Run frequency:** Once a month (or before each expiry cycle)

**What this does:**
1. Logs in to Zerodha KiteConnect via automated TOTP
2. Downloads all NFO instruments from the KiteConnect API
3. Filters BankNifty & Nifty options and futures
4. Calculates current/next weekly and monthly expiry dates
5. Writes everything to Parquet files in `DataFiles/Instruments/`

**Next step:** After this completes, run `2_Fetch_Strikes_Data.ipynb`

In [8]:
# ── Setup: add PROD/ to path so utils/ is importable ──────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [9]:
# ── Step 1: Login to KiteConnect ───────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print("✅ Login successful")

2026-03-11 09:18:50,221 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-11 09:18:52,458 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-11 09:18:52,781 [INFO] utils.kite_helpers — Login successful — request_token: vbtz0tYfUduCgk0A2t4KqWK4q87XPJBn
2026-03-11 09:18:53,018 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [10]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name="1_Get_Instruments")
print("✅ Spark session ready")

2026-03-11 09:18:53,042 [INFO] py4j.clientserver — Error while sending or receiving.
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
BrokenPipeError: [Errno 32] Broken pipe
2026-03-11 09:18:53,050 [INFO] py4j.clientserver — Closing down clientserver connection
2026-03-11 09:18:53,052 [INFO] root — Exception while sending command.
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send

ConnectionRefusedError: [Errno 61] Connection refused

2026-03-11 09:18:53,682 [INFO] py4j.clientserver — Error while sending or receiving.
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
BrokenPipeError: [Errno 32] Broken pipe
2026-03-11 09:18:53,685 [INFO] py4j.clientserver — Closing down clientserver connection
2026-03-11 09:18:53,685 [INFO] root — Exception while sending command.
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send

In [ ]:
# ── Step 3: Fetch all instruments from KiteConnect ─────────────────────────
all_instruments = kite.instruments()
print(f"Total instruments fetched: {len(all_instruments)}")
print("Sample:", all_instruments[0])

Total instruments fetched: 160385
Sample: {'instrument_token': 211574533, 'exchange_token': '826463', 'tradingsymbol': 'BANKEX26MARFUT', 'name': 'BANKEX', 'last_price': 0.0, 'expiry': datetime.date(2026, 3, 25), 'strike': 0.0, 'tick_size': 0.05, 'lot_size': 30, 'instrument_type': 'FUT', 'segment': 'BFO-FUT', 'exchange': 'BFO'}


In [ ]:
# ── Step 4: Filter BankNifty & Nifty instruments ───────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import asc, min as spark_min
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from utils.strike_utils import INSTRUMENT_SCHEMA, EXPIRY_SCHEMA

# Filter by exchange, segment, and name
bnf_options  = [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-OPT' and i['name']=='BANKNIFTY']
nifty_options= [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-OPT' and i['name']=='NIFTY']
bnf_futures  = [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-FUT' and i['name']=='BANKNIFTY']
nifty_futures= [i for i in all_instruments if i['exchange']=='NFO' and i['segment']=='NFO-FUT' and i['name']=='NIFTY']

# Create Spark DataFrames
BNF_Options_DF     = spark.createDataFrame(bnf_options,   schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))
Nifty_Options_DF   = spark.createDataFrame(nifty_options, schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))
BNF_Futures_DF     = spark.createDataFrame(bnf_futures,   schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))
Nifty_Futures_DF   = spark.createDataFrame(nifty_futures, schema=INSTRUMENT_SCHEMA).orderBy(asc('expiry'))

print(f"BankNifty Options: {BNF_Options_DF.count()} rows")
print(f"Nifty Options:     {Nifty_Options_DF.count()} rows")
BNF_Options_DF.show(5, truncate=False)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py:154: DeprecationWarning: This process (pid=44367) is multi-threaded, use of fork() may lead to deadlocks in the child.


BankNifty Options: 1118 rows


Nifty Options:     2082 rows


+----------------+--------------+---------------------+---------+----------+----------+-------+---------+--------+---------------+-------+--------+
|instrument_token|exchange_token|tradingsymbol        |name     |last_price|expiry    |strike |tick_size|lot_size|instrument_type|segment|exchange|
+----------------+--------------+---------------------+---------+----------+----------+-------+---------+--------+---------------+-------+--------+
|13415938        |52406         |BANKNIFTY26MAR53200CE|BANKNIFTY|0.0       |2026-03-30|53200.0|0.05     |30      |CE             |NFO-OPT|NFO     |
|13433346        |52474         |BANKNIFTY26MAR56000CE|BANKNIFTY|0.0       |2026-03-30|56000.0|0.05     |30      |CE             |NFO-OPT|NFO     |
|13471234        |52622         |BANKNIFTY26MAR61700CE|BANKNIFTY|0.0       |2026-03-30|61700.0|0.05     |30      |CE             |NFO-OPT|NFO     |
|13434114        |52477         |BANKNIFTY26MAR56000PE|BANKNIFTY|0.0       |2026-03-30|56000.0|0.05     |30     

In [ ]:
# ── Step 5: Write instruments to Parquet ────────────────────────────────────
OUTPUT_BASE = 'DataFiles/Instruments'

BNF_Options_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Banknifty_Options')
Nifty_Options_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Nifty_Options')
BNF_Futures_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Banknifty_Futures')
Nifty_Futures_DF.coalesce(1).write.format('Delta').mode('Overwrite').parquet(f'{OUTPUT_BASE}/Nifty_Futures')
print('✅ Instrument Parquet files written')

26/03/11 00:33:09 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Young Generation), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/03/11 00:33:09 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Old Generation, G1 Young Generation), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


✅ Instrument Parquet files written


In [ ]:
# ── Step 6: Calculate expiry dates and write to Parquet ────────────────────
from pyspark.sql.types import StructType, StructField, StringType, DateType

def _min_expiry(df):     return df.agg(spark_min('expiry')).collect()[0][0]
def _next_expiry(df, e): return df.filter(df['expiry'] > e).agg(spark_min('expiry')).collect()[0][0]

bnf_curr_week   = _min_expiry(BNF_Options_DF)
bnf_curr_month  = _min_expiry(BNF_Futures_DF)
bnf_next_week   = _next_expiry(BNF_Options_DF, bnf_curr_week)
bnf_next_month  = _next_expiry(BNF_Futures_DF, bnf_curr_month)

nifty_curr_week  = _min_expiry(Nifty_Options_DF)
nifty_curr_month = _min_expiry(Nifty_Futures_DF)
nifty_next_week  = _next_expiry(Nifty_Options_DF, nifty_curr_week)
nifty_next_month = _next_expiry(Nifty_Futures_DF, nifty_curr_month)

expiry_data = [
    ('BANKNIFTY', bnf_curr_week,   bnf_curr_month,   bnf_next_week,   bnf_next_month),
    ('NIFTY',     nifty_curr_week, nifty_curr_month, nifty_next_week, nifty_next_month),
]

from utils.strike_utils import EXPIRY_SCHEMA
spark.createDataFrame(expiry_data, schema=EXPIRY_SCHEMA)\
     .coalesce(1).write.format('Delta').mode('Overwrite')\
     .parquet(f'{OUTPUT_BASE}/Nifty_Banknifty_Expiries')

print('✅ Expiry Parquet file written')
print(f'BankNifty  current week: {bnf_curr_week}  |  next week: {bnf_next_week}')
print(f'Nifty      current week: {nifty_curr_week}  |  next week: {nifty_next_week}')

✅ Expiry Parquet file written
BankNifty  current week: 2026-03-30  |  next week: 2026-04-28
Nifty      current week: 2026-03-10  |  next week: 2026-03-17


26/03/11 04:48:02 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 269004 ms exceeds timeout 120000 ms
26/03/11 04:48:02 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/11 04:48:09 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

---
**Done.** Parquet files are written to `DataFiles/Instruments/`.  
Proceed to `2_Fetch_Strikes_Data.ipynb` to start pulling OHLC data.